# BCV Pseudo-R² vs n_components — Reaching

Load pre-computed BCV scores from `results/sweep-n_component/` and plot
pseudo-R² as a function of `n_components` for:
- **nonlinear-nonlinear** (MLP decoder, Poisson loss)
- **nonlinear-linear** (Poisson GLM decoder)

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

## Config

In [ ]:
SWEEP_ROOT   = "./results/sweep-n_component/r2"
N_COMPONENTS = [6, 10, 16, 20, 30, 40]

CONDITIONS = {
    "nonlinear-nonlinear": {"color": "#2196F3", "label": "nonlinear-nonlinear\n(MLP, Poisson loss)"},
    "nonlinear-linear":    {"color": "#FF9800", "label": "nonlinear-linear\n(Poisson GLM)"},
}

## 1. Load BCV scores

Each `bcv_summary.pt` is a dict mapping `n_components → [fold_0_score, ..., fold_4_score]`.

In [ ]:
scores = {}

for cond in CONDITIONS:
    summary_path = os.path.join(SWEEP_ROOT, cond, "bcv_summary.pt")
    if not os.path.exists(summary_path):
        # Fall back to loading per-n_components files
        summary = {}
        for K in N_COMPONENTS:
            p = os.path.join(SWEEP_ROOT, cond, f"n_components_{K}", "bcv_scores.pt")
            if os.path.exists(p):
                summary[K] = torch.load(p, weights_only=False)
        scores[cond] = summary
    else:
        scores[cond] = torch.load(summary_path, weights_only=False)

for cond, summary in scores.items():
    print(f"{cond}:")
    for K in sorted(summary):
        vals = summary[K]
        print(f"  n_components={K:2d}: mean={np.mean(vals):.4f} ± {np.std(vals):.4f}  folds={[f'{v:.4f}' for v in vals]}")

## 2. Plot pseudo-R² vs n_components

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for cond, meta in CONDITIONS.items():
    summary = scores.get(cond, {})
    Ks      = sorted(summary.keys())
    if not Ks:
        print(f"No data for {cond}, skipping.")
        continue

    means = np.array([np.mean(summary[K]) for K in Ks])
    stds  = np.array([np.std(summary[K])  for K in Ks])

    ax.plot(Ks, means, color=meta["color"], lw=2, marker="o", ms=5, label=meta["label"])
    ax.fill_between(Ks, means - stds, means + stds, color=meta["color"], alpha=0.15)

ax.set_xlabel("n_components", fontsize=11)
ax.set_ylabel("Pseudo-R² (held-out neurons, held-out trials)", fontsize=10)
ax.set_title("BCV Pseudo-R² vs n_components — reaching (Poisson)", fontsize=11)
ax.set_xticks(N_COMPONENTS)
ax.axhline(0, color="black", lw=0.8, ls=":")
ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(ls=":", alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(SWEEP_ROOT, "R2_vs_ncomponents.pdf"), bbox_inches="tight")
plt.show()

## 3. Per-fold detail plot

In [ ]:
n_conds = len(CONDITIONS)
fig, axes = plt.subplots(1, n_conds, figsize=(6 * n_conds, 4), sharey=True)

for ax, (cond, meta) in zip(axes, CONDITIONS.items()):
    summary = scores.get(cond, {})
    Ks      = sorted(summary.keys())

    n_folds = max(len(v) for v in summary.values()) if summary else 5
    fold_colors = plt.cm.Blues(np.linspace(0.4, 0.9, n_folds)) if "nonlinear-nonlinear" in cond \
                  else plt.cm.Oranges(np.linspace(0.4, 0.9, n_folds))

    for fi in range(n_folds):
        fold_vals = [summary[K][fi] for K in Ks if fi < len(summary[K])]
        ax.plot(Ks, fold_vals, color=fold_colors[fi], lw=1, alpha=0.7, marker="o", ms=3)

    means = np.array([np.mean(summary[K]) for K in Ks])
    ax.plot(Ks, means, color=meta["color"], lw=2.5, marker="o", ms=6, label="mean")

    ax.set_xlabel("n_components", fontsize=10)
    ax.set_title(meta["label"], fontsize=9, color=meta["color"])
    ax.set_xticks(Ks)
    ax.axhline(0, color="black", lw=0.8, ls=":")
    ax.grid(ls=":", alpha=0.4)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8)

axes[0].set_ylabel("Pseudo-R² (held-out neurons, held-out trials)", fontsize=10)
plt.suptitle("BCV per-fold scores vs n_components — reaching", fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(SWEEP_ROOT, "R2_vs_ncomponents_per_fold.pdf"), bbox_inches="tight")
plt.show()